# Tái hiện benchmark FashionIQ của MagicLens

Notebook này chạy lại kết quả **FashionIQ** (R@10, R@50 trên tập val của 3 danh mục dress, shirt, toptee) bằng mã gốc trong repo [google-deepmind/magiclens](https://github.com/google-deepmind/magiclens), rồi so với Bảng 12 của bài báo.

## Trước khi chạy

**1. Chọn GPU.** Vào **Runtime → Change runtime type**, chọn **A100 GPU**, bấm **Save**.

**2. Dữ liệu FashionIQ: không cần chuẩn bị.** Nếu thư mục `MyDrive/magiclens/data/fiq/` trên Drive chưa có hoặc đang trống, ô 8 tự tải `fashion-iq.7z` (khoảng 1 GB, không cần tài khoản) từ [Yandex Disk](https://disk.yandex.com/d/Z2E54WCwvrQA3A). Đây là đường dẫn được chia sẻ trong [issue #18 của repo fashion-iq](https://github.com/XiaoxiaoGuo/fashion-iq/issues/18). Notebook lưu một bản vào thư mục đó để các phiên sau khỏi tải lại. Bản này có đủ ảnh cho tập val của cả ba danh mục.

Nếu đã có dữ liệu sẵn, bạn đặt vào thư mục đó thay vì để notebook tự tải (không cần giải nén):

- tệp `fashion-iq.7z`, hoặc ba tệp `images.tar.gz`, `image_splits….zip`, `captions….zip`;
- hoặc một tệp zip bất kỳ chứa ba thư mục `captions`, `image_splits`, `images` (ví dụ nén lại thư mục `data/fiq` của dự án cũ).

Nếu bộ ảnh bị thiếu (ví dụ tải từ đường dẫn Amazon gốc, nhiều link đã chết), notebook bỏ các ảnh và truy vấn bị thiếu rồi vẫn chạy. Khi đó kết quả không so trực tiếp được với bài báo, và bảng cuối sẽ ghi rõ điều này.

## Cách chạy

- Chạy lần lượt từng ô từ trên xuống: bấm vào ô rồi nhấn **Shift + Enter**, hoặc bấm nút ▶ bên trái ô. Muốn chạy hết một lượt thì chọn **Runtime → Run all**.
- Ô 4 sẽ hỏi quyền truy cập Google Drive: chọn tài khoản của bạn và đồng ý.
- Kết quả (bảng so sánh, log, tệp top-50) được lưu vào `MyDrive/magiclens/ket_qua/fiq/` trên Google Drive.
- Mỗi lần mở lại notebook là một máy Colab mới, nên phải chạy lại từ ô 1. Trong cùng một phiên, ô nào đã làm xong sẽ tự bỏ qua.
- Pip có thể in cảnh báo bắt đầu bằng `ERROR: pip's dependency resolver...`. Đây chỉ là cảnh báo xung đột phiên bản với các gói có sẵn của Colab, **bỏ qua được**. Notebook chỉ dừng khi có lỗi thật.
- Xong việc thì vào **Runtime → Disconnect and delete runtime** để không tốn compute unit.

## Bước 1: Cấu hình

In [ ]:
# Ô 1: Cấu hình. Chỉ cần sửa ở ô này.
MODEL_SIZE = "base"          # "base" hoặc "large"
BATCH_SIZE = None            # None = tự chọn; giảm (ví dụ 256) nếu báo RESOURCE_EXHAUSTED
DANH_MUC = ["dress", "shirt", "toptee"]
BO_QUA_ANH_THIEU = True      # True: bỏ ảnh và truy vấn bị thiếu ảnh thay vì dừng lại

DRIVE_DIR = "/content/drive/MyDrive/magiclens"   # thư mục làm việc trên Google Drive
FIQ_NGUON = DRIVE_DIR + "/data/fiq"              # nơi bạn đặt các tệp nén của FashionIQ
REPO = "/content/magiclens"                      # nơi đặt mã nguồn trên máy Colab
BENCHMARK = "fiq"

## Bước 2: Chuẩn bị máy Colab

Các ô 2–7 kiểm tra máy, kết nối Drive, cài thư viện, tải checkpoint và chạy thử mô hình. Ở ô 7, dòng cuối phải có `chạy trên: GPU`. Nếu là `CPU` thì runtime chưa có GPU hoặc JAX chưa nhận GPU: kiểm tra lại ô 3 và ô 5.

In [ ]:
# Ô 2: Các hàm dùng chung. Chạy ô này, không cần sửa gì.
import ast, codecs, glob, json, os, re, shlex, shutil, subprocess, sys, tarfile, time, zipfile

try:
    import google.colab  # có mặt khi chạy trên Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"           # bớt log của TensorFlow
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"   # TensorFlow không giữ trước bộ nhớ GPU
PY = sys.executable                                 # trình Python dùng để cài thư viện và chạy mã
CONTENT = os.path.dirname(REPO)                     # /content trên Colab
LOG_DIR = os.path.join(CONTENT, "logs")
os.makedirs(LOG_DIR, exist_ok=True)
q = shlex.quote


def sh(cmd, log=None, check=True):
    # Chạy một lệnh shell, in kết quả ngay khi có và ghi vào tệp log nếu cần.
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    dec = codecs.getincrementaldecoder("utf-8")(errors="replace")
    f = open(log, "wb") if log else None
    try:
        while True:
            chunk = p.stdout.read1(65536)
            if not chunk:
                break
            sys.stdout.write(dec.decode(chunk))
            sys.stdout.flush()
            if f:
                f.write(chunk)
    finally:
        if f:
            f.close()
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f"Lệnh bị lỗi (mã {rc}). Đọc thông báo ngay phía trên để biết nguyên nhân.")
    return rc


def chay_python(ma, cwd=None):
    # Ghi đoạn mã ra tệp rồi chạy bằng một tiến trình Python riêng.
    thu_muc = cwd or CONTENT
    tep = os.path.join(thu_muc, "_chay_thu.py")
    with open(tep, "w", encoding="utf-8") as f:
        f.write(ma)
    return sh(f"cd {q(thu_muc)} && {q(PY)} {q(tep)}")


def ket_qua_dir():
    d = os.path.join(DRIVE_DIR, "ket_qua", BENCHMARK, MODEL_SIZE)
    os.makedirs(d, exist_ok=True)
    return d


def chay_inference(dataset, ten_log, env="", output="./output"):
    # Chạy inference.py của repo cho một bộ dữ liệu, ghi log, trả về đường dẫn log.
    bs = BATCH_SIZE or (1024 if MODEL_SIZE == "base" else 512)
    log = os.path.join(LOG_DIR, ten_log)
    cmd = (f"cd {q(REPO)} && {env} {q(PY)} inference.py --model_size {MODEL_SIZE} "
           f"--model_path ./models/magic_lens_clip_{MODEL_SIZE}.pkl --dataset {dataset} "
           f"--batch_size {bs} --output {output}")
    t0 = time.time()
    sh(cmd, log=log)
    print(f"\nXong {dataset} sau {(time.time() - t0) / 60:.1f} phút. Log: {log}")
    shutil.copy(log, ket_qua_dir())
    return log


def doc_recall(log):
    # Đọc dòng "Recalls:  {10: ..., 50: ...}" mà inference.py in ra.
    text = open(log, encoding="utf-8", errors="replace").read()
    found = re.findall(r"Recalls:\s*(\{[^}]*\})", text)
    if not found:
        raise RuntimeError(f"Không thấy dòng 'Recalls:' trong {log}.")
    r = ast.literal_eval(found[-1])
    return r[10], r[50]


def bang(tieu_de, cac_dong):
    # Tạo bảng Markdown.
    dong = ["| " + " | ".join(tieu_de) + " |", "|" + "---|" * len(tieu_de)]
    dong += ["| " + " | ".join(str(x) for x in d) + " |" for d in cac_dong]
    return "\n".join(dong)


def hien_thi(md):
    try:
        from IPython.display import Markdown, display
        display(Markdown(md))
    except ImportError:
        print(md)


def con_trong_gb(path):
    return shutil.disk_usage(path).free / 1e9


print("Sẵn sàng. Đang chạy trên Colab:", IN_COLAB)

In [ ]:
# Ô 3: Kiểm tra máy Colab: GPU, RAM, ổ đĩa
if sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", check=False) != 0:
    print("\nCẢNH BÁO: không thấy GPU. Vào Runtime → Change runtime type → chọn A100 GPU → Save, rồi chạy lại từ ô 1.")
sh("free -g | head -2", check=False)
print(f"Ổ đĩa còn trống: {con_trong_gb(CONTENT):.0f} GB")
print("FashionIQ chỉ cần vài GB RAM.")

In [ ]:
# Ô 4: Kết nối Google Drive. Colab sẽ hỏi quyền truy cập: chọn tài khoản của bạn và đồng ý.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Thư mục làm việc trên Drive:", DRIVE_DIR)

In [ ]:
# Ô 5: Tải mã nguồn và cài thư viện. Lần đầu mất vài phút; chạy lại trong cùng phiên sẽ tự bỏ qua.
DA_CAI = os.path.join(CONTENT, ".magiclens_da_cai")
if os.path.exists(DA_CAI):
    print("Đã cài trong phiên này, bỏ qua.")
else:
    if not os.path.isdir(REPO):
        sh(f"git clone -q https://github.com/google-deepmind/magiclens.git {q(REPO)}")
    scenic = os.path.join(CONTENT, "scenic")
    if not os.path.isdir(scenic):
        sh(f"git clone -q https://github.com/google-research/scenic.git {q(scenic)}")
    pip = f"{q(PY)} -m pip install -q"
    sh(f"{pip} {q(scenic)}")
    sh(f"{pip} -r {q(scenic + '/scenic/projects/baselines/clip/requirements.txt')}")
    # Bộ phiên bản đã chạy được với mã này. Nếu không cài được thì dùng bản mới nhất.
    if sh(f'{pip} "jax[cuda12]==0.11.1" "flax==0.12.9"', check=False) != 0:
        print("Không cài được bộ phiên bản đã kiểm thử, chuyển sang bản mới nhất.")
        sh(f'{pip} -U "jax[cuda12]" flax')
    open(DA_CAI, "w").close()
chay_python("import jax, flax\nprint('jax', jax.__version__, '| flax', flax.__version__, '|', jax.devices())\n")

In [ ]:
# Ô 6: Tải checkpoint MagicLens (công khai, không cần đăng nhập). Bản base 666 MB, bản large 1.9 GB.
KICH_THUOC = {"base": 665747853, "large": 1861692696}
CKPT = os.path.join(REPO, "models", f"magic_lens_clip_{MODEL_SIZE}.pkl")
if os.path.exists(CKPT) and os.path.getsize(CKPT) == KICH_THUOC[MODEL_SIZE]:
    print("Đã có checkpoint:", CKPT)
else:
    os.makedirs(os.path.dirname(CKPT), exist_ok=True)
    url = f"https://storage.googleapis.com/gresearch/magiclens/models/magic_lens_clip_{MODEL_SIZE}.pkl"
    sh(f"wget -q --show-progress -c -O {q(CKPT)} {url}")
    if os.path.getsize(CKPT) != KICH_THUOC[MODEL_SIZE]:
        raise RuntimeError("Tệp tải về chưa đủ dung lượng. Chạy lại ô này để tải tiếp.")
    print("Đã tải:", CKPT)

In [ ]:
# Ô 7: Chạy thử mô hình một lần để chắc mọi thứ đã sẵn sàng
chay_python(
    "import jax, jax.numpy as jnp\n"
    "from inference import load_model\n"
    f"model, params = load_model('{MODEL_SIZE}', 'models/magic_lens_clip_{MODEL_SIZE}.pkl')\n"
    "out = model.apply(params, {'ids': jnp.ones((1, 1, 77), jnp.int32), 'image': jnp.ones((1, 224, 224, 3))})\n"
    "print('Embedding:', out['multimodal_embed_norm'].shape, '| chạy trên:', jax.devices()[0].platform.upper())\n",
    cwd=REPO)

## Bước 3: Dữ liệu

Ô 8 lấy dữ liệu từ Drive (hoặc tự tải nếu Drive chưa có), giải nén vào máy Colab và sắp lại thành `data/fiq/captions`, `data/fiq/image_splits`, `data/fiq/images` đúng như mã yêu cầu. Ô 9 kiểm tra từng danh mục có đủ ảnh không.

In [ ]:
# Ô 8: Lấy dữ liệu FashionIQ (từ Drive, hoặc tự tải nếu Drive chưa có) rồi giải nén vào máy Colab
import urllib.parse, urllib.request
FIQ_DIR = os.path.join(REPO, "data", "fiq")
DA_GIAI_NEN = os.path.join(FIQ_DIR, ".da_giai_nen")
YANDEX = "https://disk.yandex.com/d/Z2E54WCwvrQA3A"   # đường dẫn trong issue #18 của repo fashion-iq
DUOI_NEN = (".zip", ".tar", ".tar.gz", ".tgz", ".7z")


def tai_tu_yandex(dich):
    # API công khai của Yandex Disk trả về kích thước tệp và một đường dẫn tải trực tiếp (không cần tài khoản)
    api = "https://cloud-api.yandex.net/v1/disk/public/resources"
    khoa = "?public_key=" + urllib.parse.quote(YANDEX, safe="")
    kich_thuoc = json.load(urllib.request.urlopen(api + khoa, timeout=60))["size"]
    href = json.load(urllib.request.urlopen(api + "/download" + khoa, timeout=60))["href"]
    sh(f"wget -q --show-progress -c -O {q(dich)} {q(href)}")
    if os.path.getsize(dich) != kich_thuoc:
        raise RuntimeError(f"Tải chưa đủ ({os.path.getsize(dich)}/{kich_thuoc} byte). Chạy lại ô này để tải tiếp.")


def giai_nen(nguon, dich):
    ten = os.path.basename(nguon).lower()
    if ten.endswith(".zip"):
        sh(f"unzip -q -o {q(nguon)} -d {q(dich)}")
    elif ten.endswith((".tar", ".tar.gz", ".tgz")):
        sh(f"tar -xf {q(nguon)} -C {q(dich)}")
    elif ten.endswith(".7z"):
        if shutil.which("7z"):
            sh(f"7z x -y -bso0 -bsp0 -o{q(dich)} {q(nguon)}")
        else:
            sh(f"{q(PY)} -m pip install -q py7zr && {q(PY)} -m py7zr x {q(nguon)} {q(dich)}")
    else:
        return False
    return True


def giai_nen_long_nhau(thu_muc):
    # Tệp nén nằm trong tệp nén (fashion-iq.7z chứa images.tar.gz, ...): giải nén tại chỗ rồi xóa tệp nén
    while True:
        con = [p for p in glob.glob(os.path.join(thu_muc, "**", "*"), recursive=True)
               if os.path.isfile(p) and p.lower().endswith(DUOI_NEN) and "__MACOSX" not in p]
        if not con:
            return
        for p in con:
            print("Giải nén", os.path.relpath(p, thu_muc), flush=True)
            giai_nen(p, os.path.dirname(p))
            os.remove(p)


def gom_tep(goc, mau, dich):
    # Chuyển mọi tệp khớp mẫu (tìm cả trong thư mục con của goc) vào thư mục dich.
    os.makedirs(dich, exist_ok=True)
    n = 0
    for p in glob.glob(os.path.join(goc, "**", mau), recursive=True):
        if "__MACOSX" in p or os.path.basename(p).startswith("._"):
            continue
        shutil.move(p, os.path.join(dich, os.path.basename(p)))
        n += 1
    return n


if os.path.exists(DA_GIAI_NEN):
    print("Dữ liệu đã được giải nén trong phiên này, bỏ qua.")
else:
    os.makedirs(FIQ_NGUON, exist_ok=True)
    cac_nguon = [os.path.join(FIQ_NGUON, ten) for ten in sorted(os.listdir(FIQ_NGUON))]
    tai_ve = os.path.join(CONTENT, "fashion-iq.7z")
    if not cac_nguon:
        print(f"{FIQ_NGUON} đang trống: tải fashion-iq.7z (khoảng 1 GB) từ Yandex Disk.", flush=True)
        tai_tu_yandex(tai_ve)
        cac_nguon = [tai_ve]
        try:
            shutil.copy(tai_ve, FIQ_NGUON)   # lưu một bản lên Drive để các phiên sau khỏi tải lại
            print("Đã lưu một bản vào", FIQ_NGUON)
        except OSError as e:
            print("Không lưu được lên Drive, vẫn dùng bản vừa tải:", e)
    tam = os.path.join(CONTENT, "fiq_tam")
    os.makedirs(tam, exist_ok=True)
    for p in cac_nguon:
        ten = os.path.basename(p)
        if os.path.isdir(p):
            print("Chép thư mục", ten, "(thư mục nhiều ảnh sẽ chép chậm; nên nén thành một tệp)")
            shutil.copytree(p, os.path.join(tam, ten), dirs_exist_ok=True)
        elif ten.endswith(".json"):
            shutil.copy(p, tam)
        elif giai_nen(p, tam):
            print("Đã giải nén", ten, flush=True)
        else:
            print("Bỏ qua tệp không phải tệp nén:", ten)
    giai_nen_long_nhau(tam)
    if os.path.exists(tai_ve):
        os.remove(tai_ve)
    n_cap = gom_tep(tam, "cap.*.json", os.path.join(FIQ_DIR, "captions"))
    n_spl = gom_tep(tam, "split.*.json", os.path.join(FIQ_DIR, "image_splits"))
    n_img = gom_tep(tam, "*.png", os.path.join(FIQ_DIR, "images"))
    shutil.rmtree(tam)
    print(f"captions: {n_cap} tệp | image_splits: {n_spl} tệp | images: {n_img} ảnh")
    if not (n_cap and n_spl and n_img):
        raise RuntimeError("Thiếu captions, image_splits hoặc ảnh .png. Kiểm tra lại các tệp trong " + FIQ_NGUON)
    open(DA_GIAI_NEN, "w").close()

In [ ]:
# Ô 9: Kiểm tra đủ ảnh. Nếu thiếu, bỏ ảnh và truy vấn liên quan (khi BO_QUA_ANH_THIEU = True).
co_anh = {f[:-4] for f in os.listdir(os.path.join(FIQ_DIR, "images")) if f.endswith(".png")}
CHAY, DA_LOC, cac_dong = [], set(), []
for t in DANH_MUC:
    cap_p = os.path.join(FIQ_DIR, "captions", f"cap.{t}.val.json")
    spl_p = os.path.join(FIQ_DIR, "image_splits", f"split.{t}.val.json")
    if not (os.path.exists(cap_p) and os.path.exists(spl_p)):
        print(f"{t}: không có tệp captions hoặc image_splits của tập val, bỏ qua.")
        continue
    # Bản gốc được giữ ở *.goc.json khi ô này phải lọc dữ liệu
    cap_goc, spl_goc = cap_p[:-5] + ".goc.json", spl_p[:-5] + ".goc.json"
    caps = json.load(open(cap_goc if os.path.exists(cap_goc) else cap_p))
    split = json.load(open(spl_goc if os.path.exists(spl_goc) else spl_p))
    split_ok = [i for i in split if i in co_anh]
    caps_ok = [c for c in caps if c["candidate"] in co_anh and c["target"] in co_anh]
    cac_dong.append([t, f"{len(caps_ok)}/{len(caps)}", f"{len(split_ok)}/{len(split)}"])
    if len(split_ok) == len(split) and len(caps_ok) == len(caps):
        for goc, moi in [(cap_goc, cap_p), (spl_goc, spl_p)]:
            if os.path.exists(goc):
                shutil.copy(goc, moi)    # trả lại bản gốc nếu trước đó đã lọc
        CHAY.append(t)
        continue
    if not caps_ok:
        print(f"{t}: không có truy vấn nào đủ ảnh, bỏ qua danh mục này.")
        continue
    if not BO_QUA_ANH_THIEU:
        raise RuntimeError(f"{t}: thiếu {len(split) - len(split_ok)} ảnh. Bổ sung ảnh, hoặc đặt BO_QUA_ANH_THIEU = True ở ô 1.")
    for goc, moi, du_lieu in [(cap_goc, cap_p, caps_ok), (spl_goc, spl_p, split_ok)]:
        if not os.path.exists(goc):
            shutil.copy(moi, goc)        # giữ bản gốc
        with open(moi, "w") as f:
            json.dump(du_lieu, f)
    DA_LOC.add(t)
    CHAY.append(t)

hien_thi(bang(["Danh mục", "Truy vấn dùng được", "Ảnh dùng được trong kho"], cac_dong))
print("Bản đầy đủ: dress 2017 truy vấn / 3817 ảnh, shirt 2038 / 6346, toptee 1961 / 5373.")
if DA_LOC:
    print("CẢNH BÁO: đã bỏ ảnh thiếu ở", ", ".join(sorted(DA_LOC)), "nên kết quả các danh mục này không so trực tiếp được với bài báo.")
if not CHAY:
    raise RuntimeError("Không có danh mục nào chạy được.")
print("Sẽ chạy:", ", ".join(CHAY))

## Bước 4: Chạy đánh giá

Mỗi danh mục chạy `inference.py` một lần. Mã nạp toàn bộ ảnh vào RAM, mã hóa kho ảnh, rồi in ra `Recalls: {10: ..., 50: ...}`.

In [ ]:
# Ô 10: Chạy đánh giá từng danh mục
KET_QUA = {}
for t in CHAY:
    log = chay_inference(f"fiq-{t}", f"fiq-{t}_{MODEL_SIZE}.log")
    KET_QUA[t] = doc_recall(log)
    print(f"==> {t}: R@10 = {KET_QUA[t][0]:.2f}, R@50 = {KET_QUA[t][1]:.2f}\n")

## Bước 5: So sánh với bài báo

In [ ]:
# Ô 11: So sánh với Bảng 12 của bài báo và lưu lên Drive
BAI_BAO = {
    "base":  {"dress": (21.5, 41.3), "shirt": (27.3, 48.8), "toptee": (30.2, 52.3), "trung bình": (26.3, 47.4)},
    "large": {"dress": (25.5, 46.1), "shirt": (32.7, 53.8), "toptee": (34.0, 57.7), "trung bình": (30.7, 52.5)},
}[MODEL_SIZE]


def dong_ket_qua(ten, r10, r50, ghi_chu=""):
    b10, b50 = BAI_BAO[ten]
    return [ten + ghi_chu, f"{r10:.1f} / {r50:.1f}", f"{b10} / {b50}", f"{r10 - b10:+.1f} / {r50 - b50:+.1f}"]


cac_dong = [dong_ket_qua(t, *KET_QUA[t], " (đã bỏ ảnh thiếu)" if t in DA_LOC else "") for t in KET_QUA]
if len(KET_QUA) == 3:
    tb10 = sum(v[0] for v in KET_QUA.values()) / 3
    tb50 = sum(v[1] for v in KET_QUA.values()) / 3
    cac_dong.append(dong_ket_qua("trung bình", tb10, tb50, " (có danh mục đã bỏ ảnh thiếu)" if DA_LOC else ""))
md_kq = (f"### FashionIQ, MagicLens-{MODEL_SIZE} (R@10 / R@50)\n\n"
         + bang(["Danh mục", "Tái hiện", "Bài báo", "Chênh lệch"], cac_dong))
hien_thi(md_kq)
with open(os.path.join(ket_qua_dir(), "tong_hop.md"), "w", encoding="utf-8") as f:
    f.write(md_kq + "\n")
with open(os.path.join(ket_qua_dir(), "ket_qua.json"), "w") as f:
    json.dump({t: {"R@10": v[0], "R@50": v[1], "da_bo_anh_thieu": t in DA_LOC} for t, v in KET_QUA.items()}, f, indent=1)
print("Đã lưu vào", ket_qua_dir())
if DA_LOC:
    print("Danh mục đã bỏ ảnh thiếu có kho ảnh nhỏ hơn bản gốc, nên điểm thường cao hơn bài báo.")
else:
    print("Checkpoint công bố là bản chuyển đổi, nên lệch dưới khoảng 1 điểm là bình thường.")

## Chạy bản Large

Ở ô 1, đổi `MODEL_SIZE = "large"`, chạy lại ô 1, rồi chạy tiếp từ ô 6 trở xuống. Dữ liệu đã chuẩn bị trong phiên này được dùng lại. Với FashionIQ, chạy lại cả ô 9 để chọn lại các danh mục.

## Kết thúc

Kết quả đã nằm trên Drive. Để không tốn compute unit, vào **Runtime → Disconnect and delete runtime**, hoặc đổi `NGAT_KET_NOI = True` ở ô dưới rồi chạy ô đó.

In [ ]:
# Ô cuối: Ngắt kết nối máy Colab. Mọi thứ trên máy sẽ bị xóa, kết quả trên Drive vẫn còn.
NGAT_KET_NOI = False   # đổi thành True rồi chạy ô này khi đã xong
if NGAT_KET_NOI and IN_COLAB:
    from google.colab import runtime
    runtime.unassign()